# TCGA-BRCA: Histopathology Image Quality Assessment (IQA) & Adaptive Routing
### Modality: Histopathology (Whole Slide Imaging & Diagnostic Patches)
### Dataset: The Cancer Genome Atlas Breast Invasive Carcinoma (TCGA-BRCA)

---

## 📌 Clinical & Analytical Objectives
1. **Dataset Characterization:** Map cohort distribution, diagnostic classes (Invasive Ductal Carcinoma `IDC`, Invasive Lobular Carcinoma `ILC`, Normal/Benign), and slide preparation types (`FFPE` vs. `Frozen Section`).
2. **Identification of Challenges:** Detect real-world pathology artifacts (freezing chatter, knife marks, air bubbles, defocus blur, H&E staining variations).
3. **Multi-Metric IQA Pipeline:** Implement No-Reference metrics:
   - **Sharpness:** Laplacian Variance ($\sigma^2_{\Delta}$), Tenengrad Gradient
   - **Contrast & Dynamic Range:** RMS Contrast ($C_{\text{RMS}}$), Michelson Contrast
   - **Information Content:** Shannon Information Entropy ($H$)
   - **Noise Estimation:** Blind SNR via Median Absolute Deviation (MAD)
   - **Perceptual Quality:** Natural Scene Statistics (MSCN distribution fitting)
4. **Statistical Analysis:** Generate descriptive summaries, metric correlations, and visual montages contrasting high-quality vs degraded slides.
5. **Learned Decision Tree:** Train an explainable multi-tier decision tree combining IQA indicators and CNN confidence to adaptively route images:
   - `Accept_Direct_Prediction`: High quality + high confidence $\to$ Autonomous diagnostic reporting.
   - `Route_Image_Enhancement`: Mild noise/contrast degradation $\to$ Automated CLAHE / Denoising pipeline.
   - `Expert_Human_Review`: Borderline quality or model uncertainty $\to$ Escalate to senior pathologist.
   - `Reject_Reacquire`: Severe blur or blank glass slide $\to$ Re-stain or re-scan request.

---
## 🚀 Running on Kaggle
- Click **"Add Input"** on the right sidebar and search for `TCGA-BRCA` or `breast-histopathology-images`.
- The notebook auto-detects the input path under `/kaggle/input/` and creates outputs in `/kaggle/working/`.


In [ ]:
# Cell 1: Environment Setup & Library Imports
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
from scipy.stats import entropy, gennorm
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Configure visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
print("Libraries loaded successfully.")


In [ ]:
# Cell 2: Kaggle Dataset Discovery & Auto-Configuration
KAGGLE_CANDIDATE_PATHS = [
    "/kaggle/input/tcga-brca",
    "/kaggle/input/tcga-brca-dataset",
    "/kaggle/input/breast-histopathology-images",
    "/kaggle/input/idc-regular-dataset",
    "./sample_data",
    "../sample_data",
]

def locate_tcga_dataset():
    for p in KAGGLE_CANDIDATE_PATHS:
        if os.path.exists(p) and len(os.listdir(p)) > 0:
            print(f"[SUCCESS] Dataset located at: {p}")
            return p
    print("[NOTICE] No external Kaggle dataset mounted. Initializing sample histology cohort...")
    return None

DATASET_ROOT = locate_tcga_dataset()
OUTPUT_DIR = "/kaggle/working/output" if os.path.exists("/kaggle") else "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Cell 3: Synthetic Cohort Fallback (Enables Zero-Error Standalone Runs)
def generate_synthetic_histology_samples(target_dir, n_samples=60):
    os.makedirs(target_dir, exist_ok=True)
    classes = ["IDC_Malignant", "ILC_Malignant", "Normal_Benign"]
    preparations = ["FFPE", "Frozen_Section"]
    np.random.seed(42)

    for i in range(n_samples):
        cls = classes[i % len(classes)]
        prep = preparations[(i // len(classes)) % len(preparations)]
        cls_dir = os.path.join(target_dir, cls)
        os.makedirs(cls_dir, exist_ok=True)

        h, w = 256, 256
        img = np.ones((h, w, 3), dtype=np.uint8)
        img[:, :, 0] = np.random.randint(200, 230)  # B
        img[:, :, 1] = np.random.randint(170, 200)  # G
        img[:, :, 2] = np.random.randint(220, 245)  # R (Eosin pink stroma)

        # Draw connective tissue fibers
        for _ in range(25):
            pt1 = (np.random.randint(0, w), np.random.randint(0, h))
            pt2 = (np.random.randint(0, w), np.random.randint(0, h))
            cv2.line(img, pt1, pt2, (180, 140, 210), 1)

        # Draw Hematoxylin purple nuclei
        n_nuclei = 120 if cls == "IDC_Malignant" else (60 if cls == "ILC_Malignant" else 30)
        for _ in range(n_nuclei):
            cx = np.random.randint(15, w - 15)
            cy = np.random.randint(15, h - 15)
            r = np.random.randint(3, 7)
            cv2.circle(img, (cx, cy), r, (130, 40, 80), -1)

        # Add Frozen Section blur / knife lines
        if prep == "Frozen_Section":
            k = np.random.choice([3, 5, 7])
            img = cv2.GaussianBlur(img, (k, k), 0)
            if np.random.rand() > 0.5:
                cv2.line(img, (0, np.random.randint(20, h - 20)), (w, np.random.randint(20, h - 20)), (240, 240, 240), 2)

        cv2.imwrite(os.path.join(cls_dir, f"TCGA_BRCA_{prep}_{cls}_{i:03d}.png"), img)
    return target_dir

if DATASET_ROOT is None:
    DATASET_ROOT = generate_synthetic_histology_samples(os.path.join(OUTPUT_DIR, "synthetic_tcga"))


## 1. Dataset Characterization & Metadata Loading
We parse file paths, assign ground truth diagnostic categories (`IDC`, `ILC`, `Normal`), and label slide preparation formats (`FFPE` vs. `Frozen Section`).


In [ ]:
# Cell 4: Load Dataset Metadata & Compute Class Balance
image_extensions = ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff")
all_paths = []
for ext in image_extensions:
    all_paths.extend(glob.glob(os.path.join(DATASET_ROOT, "**", ext), recursive=True))

metadata_records = []
for path in all_paths:
    fname = Path(path).name
    parent = Path(path).parent.name
    prep = "Frozen_Section" if ("frozen" in fname.lower() or "frozen" in parent.lower()) else "FFPE"
    
    label = "Normal_Benign"
    if "idc" in fname.lower() or "idc" in parent.lower():
        label = "IDC_Malignant"
    elif "ilc" in fname.lower() or "ilc" in parent.lower():
        label = "ILC_Malignant"
    elif "malignant" in fname.lower() or "malignant" in parent.lower():
        label = "IDC_Malignant"

    metadata_records.append({
        "image_id": fname,
        "file_path": path,
        "ground_truth_label": label,
        "preparation_type": prep
    })

df_meta = pd.DataFrame(metadata_records)
print(f"Total Cohort Images: {len(df_meta)}")
print("\nClass Distribution:")
print(df_meta["ground_truth_label"].value_counts())
print("\nPreparation Format Distribution:")
print(df_meta["preparation_type"].value_counts())


## 2. Image Quality Assessment (IQA) Implementation
We implement mathematically rigorous IQA metrics tailored for histopathology:
- **Laplacian Variance:** $\sigma^2_{\Delta} = \text{Var}(\nabla^2 I)$
- **Tenengrad Gradient:** $T = \frac{1}{N} \sum (G_x^2 + G_y^2)$
- **RMS Contrast:** $C_{\text{RMS}} = \sigma(I / 255)$
- **Michelson Contrast:** $(I_{\max} - I_{\min}) / (I_{\max} + I_{\min})$
- **Shannon Entropy:** $H = -\sum p_k \log_2 p_k$
- **Blind SNR (MAD):** $\sigma_n = \text{median}(|\nabla^2 I|) / 0.6745$, $\text{SNR}_{\text{dB}} = 20 \log_{10}(\mu / \sigma_n)$


In [ ]:
# Cell 5: IQA Metric Implementations
def compute_laplacian_variance(gray):
    return float(np.var(cv2.Laplacian(gray, cv2.CV_64F)))

def compute_tenengrad(gray):
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return float(np.mean(gx**2 + gy**2))

def compute_rms_contrast(gray):
    norm = gray.astype(np.float64) / 255.0
    return float(np.std(norm))

def compute_michelson_contrast(gray):
    p1 = float(np.percentile(gray, 1))
    p99 = float(np.percentile(gray, 99))
    return float((p99 - p1) / (p99 + p1 + 1e-6))

def compute_shannon_entropy(gray):
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256), density=True)
    hist = hist[hist > 0]
    return float(entropy(hist, base=2))

def compute_snr_mad(gray):
    mean_val = float(np.mean(gray))
    lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
    mad = float(np.median(np.abs(lap - np.median(lap))))
    sigma_n = mad / 0.6745
    if sigma_n < 1e-6:
        return 45.0
    snr = mean_val / (sigma_n + 1e-6)
    return float(np.clip(20.0 * np.log10(max(snr, 1e-6)), -10.0, 50.0))

def extract_histology_iqa(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    lap = compute_laplacian_variance(gray)
    ten = compute_tenengrad(gray)
    rms = compute_rms_contrast(gray)
    mic = compute_michelson_contrast(gray)
    ent = compute_shannon_entropy(gray)
    snr = compute_snr_mad(gray)
    
    # Composite Quality Score [0, 1]
    norm_lap = np.clip(np.log1p(lap) / 8.0, 0.0, 1.0)
    norm_contrast = np.clip(rms * 4.0, 0.0, 1.0)
    norm_snr = np.clip((snr - 5.0) / 25.0, 0.0, 1.0)
    norm_ent = np.clip(ent / 8.0, 0.0, 1.0)
    composite = float(0.30 * norm_lap + 0.25 * norm_contrast + 0.25 * norm_snr + 0.20 * norm_ent)
    
    tier = "High_Quality" if composite >= 0.65 else ("Medium_Quality" if composite >= 0.45 else "Degraded")
    
    return {
        "laplacian_variance": lap,
        "tenengrad_gradient": ten,
        "rms_contrast": rms,
        "michelson_contrast": mic,
        "shannon_entropy": ent,
        "snr_db": snr,
        "composite_quality_score": composite,
        "quality_tier": tier
    }


In [ ]:
# Cell 6: Execute IQA Feature Extraction
sample_size = min(len(df_meta), 300)
eval_df = df_meta.head(sample_size).copy()

iqa_records = []
for idx, row in eval_df.iterrows():
    metrics = extract_histology_iqa(row["file_path"])
    if metrics:
        metrics["image_id"] = row["image_id"]
        iqa_records.append(metrics)

df_iqa = pd.DataFrame(iqa_records)
df_full = pd.merge(eval_df, df_iqa, on="image_id")

# Simulate CNN model confidence (correlated with image quality)
np.random.seed(42)
df_full["cnn_confidence"] = np.clip(0.55 + 0.35 * df_full["composite_quality_score"] + np.random.normal(0, 0.04, len(df_full)), 0.40, 0.99)

print("IQA Feature Extraction Completed:")
df_full[["ground_truth_label", "preparation_type", "laplacian_variance", "rms_contrast", "shannon_entropy", "snr_db", "composite_quality_score"]].head()


## 3. Statistical Analysis & Visual Results
We present statistical summaries grouped by class and preparation type, followed by distribution plots, correlation matrices, and visual slide comparisons.


In [ ]:
# Cell 7: Generate Statistical Summary & Diagnostic Plots
print("--- TCGA-BRCA IQA Descriptive Statistics ---")
display(df_full[["laplacian_variance", "tenengrad_gradient", "rms_contrast", "shannon_entropy", "snr_db", "composite_quality_score"]].describe())

# 1. Class Distribution Plot
plt.figure(figsize=(8, 4))
sns.countplot(data=df_full, x="ground_truth_label", hue="preparation_type", palette="Set2")
plt.title("TCGA-BRCA: Cohort Class Distribution & Preparation Type", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "01_tcga_class_distribution.png"), dpi=200)
plt.show()

# 2. Quality Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.histplot(df_full["laplacian_variance"], kde=True, ax=axes[0, 0], color="crimson")
axes[0, 0].set_title("Sharpness (Laplacian Variance)", fontweight="bold")

sns.histplot(df_full["rms_contrast"], kde=True, ax=axes[0, 1], color="teal")
axes[0, 1].set_title("RMS Contrast", fontweight="bold")

sns.histplot(df_full["shannon_entropy"], kde=True, ax=axes[1, 0], color="purple")
axes[1, 0].set_title("Shannon Information Entropy (bits)", fontweight="bold")

sns.histplot(df_full["snr_db"], kde=True, ax=axes[1, 1], color="darkorange")
axes[1, 1].set_title("Signal-to-Noise Ratio (dB)", fontweight="bold")
plt.suptitle("TCGA-BRCA: Image Quality Assessment Metric Distributions", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "02_tcga_metric_distributions.png"), dpi=200)
plt.show()

# 3. FFPE vs Frozen Section Artifact Comparison
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_full, x="preparation_type", y="laplacian_variance", palette="Pastel1")
plt.title("Sharpness Degradation in Frozen Sections vs. FFPE", fontsize=12, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "03_ffpe_vs_frozen_sharpness.png"), dpi=200)
plt.show()

# 4. Best vs Degraded Slides Visual Montage
sorted_df = df_full.sort_values(by="composite_quality_score", ascending=False)
best_rows = sorted_df.head(3)
worst_rows = sorted_df.tail(3)

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for i, (_, row) in enumerate(best_rows.iterrows()):
    im = cv2.cvtColor(cv2.imread(row["file_path"]), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(im)
    axes[0, i].set_title(f"HIGH QUALITY\nScore: {row['composite_quality_score']:.2f} | Lap: {row['laplacian_variance']:.1f}", color="green", fontweight="bold")
    axes[0, i].axis("off")

for i, (_, row) in enumerate(worst_rows.iterrows()):
    im = cv2.cvtColor(cv2.imread(row["file_path"]), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(im)
    axes[1, i].set_title(f"DEGRADED / BLUR\nScore: {row['composite_quality_score']:.2f} | Lap: {row['laplacian_variance']:.1f}", color="red", fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle("Visual Comparison: High-Fidelity vs Defocused/Artifact Histology Slides", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(OUTPUT_DIR, "04_best_vs_worst_montage.png"), dpi=200)
plt.show()


## 4. Learned Decision Tree: Quality-Aware Adaptive Clinical Routing
We train an explainable Decision Tree classifier that fuses the **IQA feature vector** with **CNN prediction confidence** to make automated, safety-critical clinical routing decisions.


In [ ]:
# Cell 8: Train Learned Decision Tree & Export Routing Metadata
def assign_clinical_routing_ground_truth(row):
    q = row["composite_quality_score"]
    conf = row["cnn_confidence"]
    lap = row["laplacian_variance"]
    contrast = row["rms_contrast"]
    snr = row["snr_db"]
    
    if q < 0.25 or lap < 18.0:
        return "Reject_Reacquire"
    elif (contrast < 0.12 or snr < 11.0) and lap >= 25.0:
        return "Route_Image_Enhancement"
    elif q >= 0.60 and conf >= 0.80:
        return "Accept_Direct_Prediction"
    else:
        return "Expert_Human_Review"

df_full["clinical_decision_target"] = df_full.apply(assign_clinical_routing_ground_truth, axis=1)

feature_cols = ["composite_quality_score", "laplacian_variance", "rms_contrast", "shannon_entropy", "snr_db", "cnn_confidence"]
X = df_full[feature_cols].values
y = df_full["clinical_decision_target"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y if len(np.unique(y)) > 1 else None)

dt_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=3, criterion="gini", random_state=42)
dt_model.fit(X_train, y_train)

acc = dt_model.score(X_test, y_test)
print(f"Learned Decision Tree Test Accuracy: {acc * 100:.2f}%
")

print("--- Decision Logic Rules ---")
print(export_text(dt_model, feature_names=feature_cols))

# Feature importances
imp_df = pd.DataFrame({"Feature": feature_cols, "Importance": dt_model.feature_importances_}).sort_values(by="Importance", ascending=False)
print("--- Feature Importances ---")
display(imp_df)

# Export Full Enriched Metadata CSV
output_csv = os.path.join(OUTPUT_DIR, "tcga_brca_quality_and_routing_metadata.csv")
df_full.to_csv(output_csv, index=False)
print(f"[SUCCESS] Exported enriched clinical metadata for CNN & Tree training to: {output_csv}")
